In [3]:
path = "/mnt/cluster/workspaces/mazzalore/iros/act_checkpoints/act_training_20250507/dataset_stats.pkl"

In [4]:
import os
import pandas as pd
dataset_path = "/mnt/cluster/datasets/threading_il/v3/1/dataset.csv"
dataset = pd.read_csv(dataset_path)
initial_pos = dataset.loc[0, ['abs_pos_x_t', 'abs_pos_y_t', 'abs_pos_z_t']].values
pos_cols = ['abs_pos_x_t', 'abs_pos_y_t', 'abs_pos_z_t']
dataset[pos_cols] = dataset[pos_cols].astype(float).values -initial_pos.astype(float)
dataset[["actionx", "actiony", "actionz"]] = dataset[["abs_pos_x_t", "abs_pos_y_t", "abs_pos_z_t"]].shift(-1).ffill()


In [4]:
import pickle
with open(path, 'rb') as f:
    data = pickle.load(f)
print(data)

{'action_mean': array([ 4.75975775e-05, -1.25586323e-04,  3.30613431e-04]), 'action_std': array([0.01, 0.01, 0.01]), 'qpos_mean': array([-0.22952582, -0.00968997, -0.12062167]), 'qpos_std': array([0.21266741, 0.16170799, 0.14101741]), 'depth_mean': 32.820186471044494, 'depth_std': 10.385221481323242}


In [2]:
import os

dataset_dir = "/mnt/cluster/datasets/threading_il/v3"
datasets_paths = [os.path.join(dataset_dir, i, "smoothed_dataset.csv") for i in os.listdir(dataset_dir)]


In [5]:
!pip install joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.7/307.7 kB 1.8 MB/s eta 0:00:00a 0:00:01


In [10]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import random
from joblib import Parallel, delayed
import contextlib

# Context manager for tqdm with joblib
@contextlib.contextmanager
def tqdm_joblib(total=None, desc="Processing", **kwargs):
    """
    Context manager to patch joblib to report into tqdm progress bar
    """
    pbar = tqdm(total=total, desc=desc, **kwargs)
    
    class TqdmBatchCompletionCallback:
        def __init__(self, batch_size=1, parallel=None):
            self.batch_size = batch_size
            self.parallel = parallel
            
        def __call__(self, *args, **kwargs):
            pbar.update(self.batch_size)
            
    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    
    try:
        yield pbar
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        pbar.close()

def _sample_from_file(path: str, values_per_file: int) -> np.ndarray:
    """
    Load one depth .npy file, flatten it, and uniformly sample up to values_per_file pixels.
    Returns a 1D np.ndarray of sampled values (length ≤ values_per_file).
    """
    try:
        depth_img = np.load(path)
    except Exception as e:
        # could log e if you want
        return np.empty((0,), dtype=np.float32)
    
    flat = depth_img.ravel()
    if flat.size > values_per_file:
        idx = np.random.choice(flat.size, values_per_file, replace=False)
        return flat[idx]
    else:
        return flat

def analyze_disparity_quantiles(datasets_paths,
                                file_sample_size=10000,
                                values_per_file=1000,
                                n_jobs=-1):
    """
    Estimate the 1% and 99% quantiles from a large dataset of depth images,
    but parallelize the per-file sampling with joblib.
    """
    # 1) Build a reservoir‐sampled list of file paths (serial)
    def path_generator():
        for dataset_path in tqdm(datasets_paths, desc="Processing datasets"):
            df = pd.read_csv(dataset_path).iloc[::6]
            for p in df["left_img"]:
                yield p.replace("framesLeft", "stereoDepth").replace(".png", ".npy")
    
    print("Sampling paths...")
    sampled_paths = []
    for i, p in enumerate(tqdm(path_generator(), desc="Sampling paths")):
        if i < file_sample_size:
            sampled_paths.append(p)
        else:
            j = random.randint(0, i)
            if j < file_sample_size:
                sampled_paths[j] = p
    
    print(f"Sampled {len(sampled_paths)} paths")
    
    # 2) Parallel load+sample values_per_file values from each path
    print("Processing files in parallel...")
    results = Parallel(n_jobs=n_jobs)(
            delayed(_sample_from_file)(path, values_per_file) 
            for path in tqdm(sampled_paths)
        )
    
    # 3) Concatenate & trim to fixed buffer size
    print("Calculating statistics...")
    all_vals = np.concatenate(results, axis=0)
    # (we might get < file_sample_size * values_per_file if loads failed or images small)
    
    # 4) Compute statistics
    q01, q99 = np.percentile(all_vals, [1, 99])
    stats = {
        'min':   all_vals.min(),
        'max':   all_vals.max(),
        'mean':  all_vals.mean(),
        'median':np.median(all_vals),
        'std':   all_vals.std(),
        'p1':    q01,
        'p99':   q99,
        'count': all_vals.size
    }
    print(f"Processed {len(sampled_paths)} files → {all_vals.size:,} samples")
    print(f"1% quantile:  {q01:.4f}")
    print(f"99% quantile: {q99:.4f}")
    return stats

# Usage
q01, q99 = analyze_disparity_quantiles(datasets_paths)

Sampling paths...


Sampling paths: 0it [00:00, ?it/s]
Sampling paths: 866it [00:00, 8102.87it/s]8 [00:00<?, ?it/s]
Sampling paths: 1994it [00:00, 9529.82it/s]8 [00:00<00:08, 122.70it/s]
Sampling paths: 2947it [00:00, 9491.08it/s]8 [00:00<00:07, 127.41it/s]
Processing datasets:   4%|▍         | 41/998 [00:00<00:07, 132.45it/s]
Sampling paths: 4788it [00:00, 8617.89it/s]8 [00:00<00:07, 130.85it/s]
Sampling paths: 5747it [00:00, 8774.84it/s]8 [00:00<00:07, 120.81it/s]
Sampling paths: 6687it [00:00, 8868.00it/s]8 [00:00<00:07, 127.23it/s]
Sampling paths: 7620it [00:00, 8842.45it/s]8 [00:00<00:06, 128.99it/s]
Sampling paths: 8720it [00:00, 9286.58it/s]98 [00:00<00:07, 125.85it/s]
Sampling paths: 9862it [00:01, 9785.65it/s]98 [00:00<00:06, 126.38it/s]
Sampling paths: 11028it [00:01, 10187.91it/s] [00:01<00:06, 127.95it/s]
Sampling paths: 12048it [00:01, 9992.97it/s]  [00:01<00:06, 126.29it/s]
Sampling paths: 13049it [00:01, 9579.81it/s]8 [00:01<00:06, 128.53it/s]
Processing datasets:  18%|█▊        | 178/998 [

Sampled 10000 paths
Processing files in parallel...


100%|██████████| 10000/10000 [05:59<00:00, 27.79it/s]


Calculating statistics...
Processed 10000 files → 10,000,000 samples
1% quantile:  15.3722
99% quantile: 63.9289


ValueError: too many values to unpack (expected 2)

In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import random

def analyze_disparity_quantiles_videos(datasets_paths, frames_per_episode=20, values_per_frame=1000):
    """
    Estimate quantiles across video episodes, sampling multiple frames per episode.
    
    Parameters:
    datasets_paths: List of paths to CSV files containing episode data
    frames_per_episode: Number of frames to sample from each episode
    values_per_frame: Maximum number of values to sample from each frame
    """
    # Create a dictionary to store all sampled values
    all_values = []
    total_frames_processed = 0
    
    # Process each dataset/episode
    for dataset_path in tqdm(datasets_paths, desc="Processing datasets"):
        try:
            # Load the dataset
            dataset = pd.read_csv(dataset_path)
            dataset = dataset.iloc[::6]  # Same sampling as your original code
            depth_paths = dataset["left_img"].apply(
                lambda x: x.replace("framesLeft", "stereoDepth").replace(".png", ".npy")
            )
            
            # Convert to list for easy indexing
            depth_paths_list = depth_paths.tolist()
            num_frames = len(depth_paths_list)
            
            if num_frames == 0:
                continue
                
            # Sample frames strategically throughout the episode
            if num_frames <= frames_per_episode:
                # If we have fewer frames than requested, use all of them
                sampled_indices = list(range(num_frames))
            else:
                # Distribute the samples across the episode
                # This ensures we sample from beginning, middle, and end
                sampled_indices = []
                
                # Add some random frames
                random_indices = random.sample(range(num_frames), frames_per_episode // 2)
                sampled_indices.extend(random_indices)
                
                # Add some evenly distributed frames to ensure coverage
                step = num_frames // (frames_per_episode // 2 + 1)
                for i in range(0, num_frames, step):
                    if len(sampled_indices) < frames_per_episode:
                        sampled_indices.append(i)
                
                # Ensure uniqueness
                sampled_indices = list(set(sampled_indices))
            
            # Process each sampled frame
            for idx in sampled_indices:
                path = depth_paths_list[idx]
                try:
                    # Load the depth image
                    depth_img = np.load(path)
                    
                    # Sample pixels from this frame
                    flat_img = depth_img.flatten()
                    valid_indices = ~np.isnan(flat_img)  # Filter out NaN values if any
                    valid_pixels = flat_img[valid_indices]
                    
                    if len(valid_pixels) == 0:
                        continue
                    
                    # Randomly sample values from this frame
                    if len(valid_pixels) > values_per_frame:
                        indices = np.random.choice(len(valid_pixels), values_per_frame, replace=False)
                        sampled = valid_pixels[indices]
                    else:
                        sampled = valid_pixels
                    
                    # Add to our collection using a memory-efficient approach
                    all_values.append(sampled)
                    total_frames_processed += 1
                    
                except Exception as e:
                    print(f"Error loading frame {path}: {e}")
                    continue
        
        except Exception as e:
            print(f"Error processing dataset {dataset_path}: {e}")
            continue
    
    # Concatenate all sampled values (more memory efficient than appending to a single array)
    all_values_array = np.concatenate(all_values)
    
    # Calculate statistics
    q05 = np.percentile(all_values_array, 5)
    q95 = np.percentile(all_values_array, 95)
    
    min_val = np.min(all_values_array)
    max_val = np.max(all_values_array)
    mean_val = np.mean(all_values_array)
    median_val = np.median(all_values_array)
    std_val = np.std(all_values_array)
    
    # Count outliers
    outliers_below = np.sum(all_values_array < q05)
    outliers_above = np.sum(all_values_array > q95)
    total_outliers = outliers_below + outliers_above
    outlier_percentage = 100 * total_outliers / len(all_values_array)
    
    # Print results
    print(f"\nStatistics from {len(datasets_paths)} episodes, {total_frames_processed} frames:")
    print(f"Total sampled values: {len(all_values_array):,}")
    print(f"Min: {min_val:.4f}")
    print(f"Max: {max_val:.4f}")
    print(f"Mean: {mean_val:.4f}")
    print(f"Median: {median_val:.4f}")
    print(f"Standard Deviation: {std_val:.4f}")
    print(f"5% quantile: {q05:.4f}")
    print(f"95% quantile: {q95:.4f}")
    print(f"Outliers below 5% quantile: {outliers_below:,} ({100 * outliers_below / len(all_values_array):.2f}%)")
    print(f"Outliers above 95% quantile: {outliers_above:,} ({100 * outliers_above / len(all_values_array):.2f}%)")
    print(f"Total outliers: {total_outliers:,} ({outlier_percentage:.2f}%)")
    
    # Optional: Analyze extreme outliers
    extreme_q001 = np.percentile(all_values_array, 0.1)
    extreme_q999 = np.percentile(all_values_array, 99.9)
    extreme_outliers = np.sum((all_values_array < extreme_q001) | (all_values_array > extreme_q999))
    print(f"Extreme outliers (outside 0.1%-99.9%): {extreme_outliers:,} ({100 * extreme_outliers / len(all_values_array):.4f}%)")
    
    return {
        'q05': q05,
        'q95': q95,
        'min': min_val,
        'max': max_val,
        'mean': mean_val,
        'median': median_val,
        'std': std_val
    }

# Usage
stats = analyze_disparity_quantiles_videos(datasets_paths, frames_per_episode=20, values_per_frame=1000)

Processing datasets: 100%|██████████| 998/998 [16:21<00:00,  1.02it/s]



Statistics from 998 episodes, 18386 frames:
Total sampled values: 18,386,000
Min: 1.9450
Max: 475.8543
Mean: 33.5525
Median: 32.7065
Standard Deviation: 11.8880
5% quantile: 17.4862
95% quantile: 51.9200
Outliers below 5% quantile: 919,299 (5.00%)
Outliers above 95% quantile: 919,300 (5.00%)
Total outliers: 1,838,599 (10.00%)
Extreme outliers (outside 0.1%-99.9%): 36,772 (0.2000%)
